# 03 — Feature Engineering

**Project:** Can Search-Trend Data Improve Short-Horizon Macro Nowcasts? A Gradient-Boosting Approach


**Notebook purpose:** Build the leakage-safe feature matrix used by the ML challenger model in `05_modeling_ml.ipynb`. All features are constructed so that the value available for predicting month *t* uses only information dated *t-1* or earlier — this is essential for a genuine nowcasting (as opposed to in-sample fitting) exercise and is a point referees scrutinize closely.

**Feature families:**
1. Lags 1–3 of the target and of each predictor
2. Rolling 3- and 6-month mean/std of the target and predictors (computed on lagged values)
3. Month-over-month percentage change of each predictor (lagged)
4. Calendar-month dummies (seasonality)

---

In [1]:
import sys
sys.path.append('..')

import pandas as pd

from src.utils import load_config, get_path
from src.features import build_feature_matrix

config = load_config('../config.yaml')
panel = pd.read_csv('../data/processed/panel_monthly.csv', parse_dates=['date'], index_col='date')

target_col = config['target']['fred_series']
predictor_cols = [c for c in panel.columns if c != target_col]
predictor_cols

['ICSA',
 'INDPRO',
 'PAYEMS',
 'file for unemployment',
 'unemployment office',
 'jobs near me']

## 1. Build the leakage-safe feature matrix

In [2]:
feature_df = build_feature_matrix(
    panel, target_col=target_col, predictor_cols=predictor_cols, n_lags=config['modeling']['n_lags']
)
print(feature_df.shape)
feature_df.head()

(140, 67)


,UNRATE,UNRATE_lag1,UNRATE_lag2,UNRATE_lag3,ICSA_lag1,ICSA_lag2,ICSA_lag3,INDPRO_lag1,INDPRO_lag2,INDPRO_lag3,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
date,,,,,,,,,,,,,,,,,,,,,
2013-03-31,7.5,7.7,8.0,7.9,353000.0,353000.0,359000.0,98.7941,98.3342,98.3101,...,True,False,False,False,False,False,False,False,False,False
2013-06-30,7.5,7.5,7.6,7.5,347750.0,347250.0,351800.0,99.1839,99.1065,99.1921,...,False,False,False,True,False,False,False,False,False,False
2013-07-31,7.3,7.5,7.5,7.6,344600.0,347750.0,347250.0,99.3897,99.1839,99.1065,...,False,False,False,False,True,False,False,False,False,False
2013-08-31,7.2,7.3,7.5,7.5,346000.0,344600.0,347750.0,99.0814,99.3897,99.1839,...,False,False,False,False,False,True,False,False,False,False
2013-09-30,7.2,7.2,7.3,7.5,333400.0,346000.0,344600.0,99.6543,99.0814,99.3897,...,False,False,False,False,False,False,True,False,False,False


## 2. Leakage sanity check

Verify programmatically that no feature column is contemporaneously identical to (or a trivial transform of) the current-period target — i.e., every feature is dated strictly before the prediction target.

In [3]:
assert target_col not in [c for c in feature_df.columns if c != target_col and target_col in c and 'lag' not in c and 'roll' not in c]
for col in predictor_cols:
    assert col not in feature_df.columns, f'{col} raw contemporaneous value leaked into features!'
print('Leakage check passed: only lagged/rolling/seasonal features are present.')

Leakage check passed: only lagged/rolling/seasonal features are present.


## 3. Feature summary and persistence

In [4]:
feature_df.describe().T

,count,mean,std,min,25%,50%,75%,max
UNRATE,140.0,4.952143,1.750004,3.400000,3.800000,4.350000,5.625000,1.480000e+01
UNRATE_lag1,140.0,4.977857,1.763787,3.400000,3.800000,4.400000,5.700000,1.480000e+01
UNRATE_lag2,140.0,5.004286,1.779818,3.400000,3.800000,4.400000,5.725000,1.480000e+01
UNRATE_lag3,140.0,5.027857,1.789380,3.400000,3.800000,4.400000,5.800000,1.480000e+01
ICSA_lag1,140.0,366281.428571,474950.963820,197500.000000,218900.000000,246750.000000,301812.500000,4.663250e+06
ICSA_lag2,140.0,367196.071429,474788.951489,197500.000000,218900.000000,247750.000000,304875.000000,4.663250e+06
ICSA_lag3,140.0,368064.285714,474659.003057,197500.000000,218900.000000,247875.000000,313575.000000,4.663250e+06
INDPRO_lag1,140.0,100.316414,2.650267,84.561900,99.365400,100.587850,101.523375,1.041004e+02
INDPRO_lag2,140.0,100.311800,2.653013,84.561900,99.347850,100.587850,101.523375,1.041004e+02
INDPRO_lag3,140.0,100.309646,2.653880,84.561900,99.214750,100.587850,101.523375,1.041004e+02


In [5]:
out_path = get_path(config['paths']['processed_dir'] + '/feature_matrix.csv')
feature_df.to_csv(out_path)
print(f'Saved feature matrix ({feature_df.shape[0]} rows x {feature_df.shape[1]} cols) to {out_path}')

Saved feature matrix (140 rows x 67 cols) to E:\Research_Projects\Economics_Bulletin\econ-ml-nowcasting\data\processed\feature_matrix.csv


---
## Notebook summary

- Constructed a fully lagged, leakage-safe feature matrix (`data/processed/feature_matrix.csv`) with lag, rolling-statistic, month-over-month, and seasonal-dummy features.
- Verified programmatically that no contemporaneous predictor value leaks into the feature set.

**Next notebooks:** `04_modeling_benchmark.ipynb` (AR/ARIMA baseline) and `05_modeling_ml.ipynb` (XGBoost challenger), both evaluated under an identical expanding-window, one-step-ahead protocol.